In [3]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')
import pandas as pd
df = pd.read_csv("/kaggle/input/datasets/harshalhonde/starbucks-reviews-dataset/reviews_data.csv")
df.head()
import glob
glob.glob("/kaggle/input/**/*.csv", recursive=True)

/kaggle/input/datasets/harshalhonde/starbucks-reviews-dataset/reviews_data.csv


['/kaggle/input/datasets/harshalhonde/starbucks-reviews-dataset/reviews_data.csv']

In [4]:
len(df)

850

In [5]:
df["Rating"].isna()

0      False
1      False
2      False
3      False
4      False
       ...  
845     True
846     True
847     True
848     True
849     True
Name: Rating, Length: 850, dtype: bool

In [6]:
df["Rating"].isna().sum()

np.int64(145)

In [7]:
df=df.dropna(subset=["Rating","Review"])

In [8]:
len(df)

705

In [9]:
neg = df[df["Rating"]<=2]
print(len(neg))
print((len(neg)/705)*100)

550
78.01418439716312


In [18]:
from transformers import pipeline
clf = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english", device=0)
clf(df["Review"].tolist())

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[{'label': 'POSITIVE', 'score': 0.9998542070388794},
 {'label': 'POSITIVE', 'score': 0.9985255599021912},
 {'label': 'NEGATIVE', 'score': 0.9965325593948364},
 {'label': 'POSITIVE', 'score': 0.6637013554573059},
 {'label': 'POSITIVE', 'score': 0.9997188448905945},
 {'label': 'NEGATIVE', 'score': 0.9990606904029846},
 {'label': 'NEGATIVE', 'score': 0.9901567697525024},
 {'label': 'NEGATIVE', 'score': 0.9022523164749146},
 {'label': 'NEGATIVE', 'score': 0.9995067119598389},
 {'label': 'NEGATIVE', 'score': 0.9926393032073975},
 {'label': 'NEGATIVE', 'score': 0.9987701773643494},
 {'label': 'NEGATIVE', 'score': 0.9945804476737976},
 {'label': 'NEGATIVE', 'score': 0.999750554561615},
 {'label': 'NEGATIVE', 'score': 0.9992485642433167},
 {'label': 'NEGATIVE', 'score': 0.9991525411605835},
 {'label': 'NEGATIVE', 'score': 0.9990468621253967},
 {'label': 'NEGATIVE', 'score': 0.9985542893409729},
 {'label': 'NEGATIVE', 'score': 0.9950994849205017},
 {'label': 'NEGATIVE', 'score': 0.9993784427642

In [19]:
result = clf(df["Review"].tolist(),truncation=True,max_length=512,batch_size=32)
len(result)

705

In [27]:
labels = [r["label"] for r in result]
df["sentiment"]=labels

705
5


In [29]:
scores=[r["score"] for r in result]
df["confidence"]=scores

In [32]:
print(labels[:5])

['POSITIVE', 'POSITIVE', 'NEGATIVE', 'POSITIVE', 'POSITIVE']


In [36]:
df[["Rating", "sentiment", "confidence"]].head(10)
df["sentiment"].value_counts()

sentiment
NEGATIVE    571
POSITIVE    134
Name: count, dtype: int64

In [35]:
pd.crosstab(df["Rating"], df["sentiment"])

sentiment,NEGATIVE,POSITIVE
Rating,,
1.0,432,19
2.0,88,11
3.0,25,8
4.0,15,24
5.0,11,72
